# YOLO11l-seg Training from Scratch (Baseline)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marjanstoimcev/DINOv3distill/blob/main/train_from_scratch.ipynb)

This notebook trains YOLO11l-seg **from scratch** (random initialization) on the TACO dataset.

**Purpose:** Baseline comparison for the DINOv3 distillation approach.

**Model:**
- YOLO11l-seg: 27.7M params, 143.0 GFLOPs

**Dataset:** TACO - Trash Annotations in Context
- 59 classes of litter/trash
- 1,499 images with polygon segmentation masks
- License: CC BY 4.0

## 1. Install Dependencies

In [ ]:
# Install ultralytics
!pip install -q ultralytics gdown

from ultralytics import YOLO
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

In [ ]:
# Additional imports
import os
from pathlib import Path
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Hyperparameters

Configure training hyperparameters here. Defaults based on [Ultralytics recommendations](https://docs.ultralytics.com/modes/train/).

In [ ]:
# ============================================================
# HYPERPARAMETERS - Modify these as needed
# ============================================================

# Training
EPOCHS = 50                 # Number of training epochs
BATCH_SIZE = 16             # Batch size (adjust for GPU memory)
LR = 0.01                   # Initial learning rate (Ultralytics default)
LRF = 0.01                  # Final LR = LR * LRF (Ultralytics default)

# Common
IMAGE_SIZE = 640            # Input image size
PATIENCE = 10               # Early stopping patience

# Model
MODEL_CONFIG = "yolo11l-seg.yaml"  # 27.7M params

print("Hyperparameters:")
print(f"  Epochs:     {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  LR:         {LR} -> {LR * LRF} (final)")
print(f"  Image size: {IMAGE_SIZE}")
print(f"  Model:      {MODEL_CONFIG}")

## 3. Dataset Setup

In [ ]:
# Download TACO dataset from Google Drive
import gdown
import zipfile

FILE_ID = "1HZBR53rs_igtr2ZlYqsLrS0uROCcTqzl"
ZIP_PATH = "taco.zip"
DATA_PATH = Path("./data/taco")

if not DATA_PATH.exists():
    print("Downloading TACO dataset from Google Drive...")
    gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", ZIP_PATH, quiet=False)
    
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('./data')
    print("Dataset ready!")
else:
    print(f"Dataset already exists at {DATA_PATH}")

!ls -la ./data/taco/

In [ ]:
# Verify dataset structure
dataset_path = DATA_PATH

print("Dataset structure:")
print(f"  Root: {dataset_path}")

total_images = 0
for split in ['train', 'valid', 'test']:
    images_path = dataset_path / split / 'images'
    labels_path = dataset_path / split / 'labels'
    
    if images_path.exists():
        num_images = len(list(images_path.glob('*.jpg'))) + len(list(images_path.glob('*.JPG'))) + len(list(images_path.glob('*.png')))
        total_images += num_images
        print(f"  {split}/images: {num_images} images")
    
    if labels_path.exists():
        num_labels = len(list(labels_path.glob('*.txt')))
        print(f"  {split}/labels: {num_labels} labels")

print(f"\nTotal images: {total_images}")

In [ ]:
# Check data.yaml
DATA_YAML = dataset_path / "data.yaml"

print(f"Using data.yaml: {DATA_YAML}")
with open(DATA_YAML, 'r') as f:
    content = f.read()
    lines = content.split('\n')
    for line in lines[:20]:
        print(line)
    print("...")

## 4. Train YOLO11l-seg from Scratch

Training from random initialization (no pretraining).

This is the **baseline** to compare against DINOv3 distillation.

In [ ]:
# Initialize model from scratch (random weights)
model = YOLO(MODEL_CONFIG)  # .yaml = build from scratch

print(f"Model: YOLO11l-seg")
print(f"Task: {model.task}")
print(f"Initialized from: Random weights (scratch)")

In [ ]:
# Training configuration
OUTPUT_DIR = "./output/from_scratch"
RUN_NAME = "yolo11l_seg_taco_scratch"

print("Training Configuration:")
print(f"  data:     {DATA_YAML}")
print(f"  epochs:   {EPOCHS}")
print(f"  imgsz:    {IMAGE_SIZE}")
print(f"  batch:    {BATCH_SIZE}")
print(f"  lr0:      {LR}")
print(f"  lrf:      {LRF}")
print(f"  patience: {PATIENCE}")
print(f"  project:  {OUTPUT_DIR}")
print(f"  name:     {RUN_NAME}")

In [ ]:
# Train the model
print("="*60)
print("Starting training from SCRATCH...")
print("="*60)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    lr0=LR,
    lrf=LRF,
    patience=PATIENCE,
    save=True,
    plots=True,
    project=OUTPUT_DIR,
    name=RUN_NAME,
)

print("="*60)
print("Training complete!")
print("="*60)

## 5. Evaluate Model

In [ ]:
# Validate the model
metrics = model.val()

print("\n" + "="*60)
print("RESULTS: YOLO11l-seg Trained from Scratch")
print("="*60)
print(f"\nBox Detection:")
print(f"  mAP50:    {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"\nInstance Segmentation:")
print(f"  mAP50:    {metrics.seg.map50:.4f}")
print(f"  mAP50-95: {metrics.seg.map:.4f}")
print("="*60)

In [ ]:
# Save metrics for comparison
import json

metrics_dict = {
    "model": "YOLO11l-seg",
    "training": "from_scratch",
    "hyperparameters": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "lrf": LRF,
        "image_size": IMAGE_SIZE,
    },
    "box_map50": float(metrics.box.map50),
    "box_map": float(metrics.box.map),
    "seg_map50": float(metrics.seg.map50),
    "seg_map": float(metrics.seg.map),
}

# Ensure output directory exists
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

with open(f"{OUTPUT_DIR}/metrics_scratch.json", "w") as f:
    json.dump(metrics_dict, f, indent=2)

print(f"Metrics saved to: {OUTPUT_DIR}/metrics_scratch.json")
print(json.dumps(metrics_dict, indent=2))

## 6. Inference Demo

In [ ]:
# Run inference on test images
import matplotlib.pyplot as plt
from PIL import Image
import random

test_images_path = dataset_path / "test" / "images"
test_images = list(test_images_path.glob("*.jpg")) + list(test_images_path.glob("*.JPG"))
test_images = test_images[:5]

print(f"Running inference on {len(test_images)} test images...")

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(1, min(5, len(test_images)), figsize=(20, 4))

if len(test_images) == 1:
    axes = [axes]

for ax, img_path in zip(axes, test_images):
    results = model.predict(str(img_path), verbose=False)
    result_img = results[0].plot()
    
    ax.imshow(result_img[:, :, ::-1])
    ax.axis('off')
    ax.set_title(img_path.name[:20] + "...")

plt.suptitle("YOLO11l-seg (From Scratch) - Predictions", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/inference_scratch.png", dpi=150, bbox_inches='tight')
plt.show()

## 7. Export Model

In [ ]:
# Export to ONNX
model.export(format="onnx", imgsz=IMAGE_SIZE, simplify=True)
print("Model exported to ONNX format")

In [ ]:
# Download the trained model (for Colab)
from google.colab import files

best_model_path = Path(OUTPUT_DIR) / RUN_NAME / "weights" / "best.pt"

if best_model_path.exists():
    print(f"Downloading {best_model_path}...")
    files.download(str(best_model_path))
else:
    print("Best model not found. Available files:")
    for f in Path(OUTPUT_DIR).rglob("*.pt"):
        print(f"  {f}")

## Summary

This notebook trained **YOLO11l-seg from scratch** (random initialization) on the TACO dataset.

**Use this as a baseline** to compare against:
- `dinov3_distillation.ipynb` - YOLO11l-seg pretrained with DINOv3 distillation

**Expected:** The distillation-pretrained model should achieve better mAP scores, especially with limited training data.